In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import glob
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import imageio
from xarray.coders import CFDatetimeCoder

# =========================================================
# Plot style (Nature Geoscience vibe)
# =========================================================

def set_plot_style():
    plt.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "DejaVu Sans"],
        "font.size": 8,
        "axes.titlesize": 8,
        "axes.labelsize": 8,
        "axes.linewidth": 0.8,
        "axes.facecolor": "white",
        "xtick.labelsize": 7,
        "ytick.labelsize": 7,
        "xtick.direction": "out",
        "ytick.direction": "out",
        "xtick.major.size": 3,
        "xtick.major.width": 0.8,
        "ytick.major.size": 3,
        "ytick.major.width": 0.8,
        "savefig.dpi": 300,
        "figure.dpi": 300,
        "figure.facecolor": "white",
        "text.color": "black",
        "axes.edgecolor": "black",
        "xtick.color": "black",
        "ytick.color": "black",
    })


# =========================================================
# Utilities
# =========================================================

def compute_edges(coord):
    coord = np.asarray(coord)
    d = np.diff(coord) / 2.0
    return np.concatenate([[coord[0] - d[0]], coord[:-1] + d, [coord[-1] + d[-1]]])


def plev_to_hPa(plev_da):
    p = plev_da.values.astype(float)
    if np.nanmax(p) > 2000:
        p_hPa = p / 100.0  # Pa -> hPa
    else:
        p_hPa = p
    return xr.DataArray(p_hPa, coords={'plev': plev_da}, dims=['plev'])


def pressure_to_height(plev_da, p0=1e5, H=7000.0):
    p = plev_da.values.astype(float)
    if np.nanmax(p) < 2000:
        p = p * 100.0
    z_km = -H * np.log(p / p0) / 1000.0
    return xr.DataArray(z_km, coords={'plev': plev_da}, dims=['plev'])


def load_hindcast(pattern, start, end):
    coder = CFDatetimeCoder(use_cftime=True)
    files = sorted(glob.glob(pattern))
    dss = []
    for f in files:
        ds = xr.open_dataset(f, decode_times=coder)
        sel = ds.sel(time=slice(start, end))
        if sel.time.size > 0:
            dss.append(sel)
    if not dss:
        raise RuntimeError(f"No hindcast data for pattern {pattern}")
    ds_all = xr.concat(dss, dim='member')
    ds_all = ds_all.mean(dim='member')
    return ds_all


def load_control(path, start, end):
    coder = CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(path, decode_times=coder)
    sel = ds.sel(time=slice(start, end))
    if sel.time.size == 0:
        raise RuntimeError(f"No control data for slice {start}–{end}")
    return sel


# =========================================================
# O3 DU/km & DU (verification)
# =========================================================

def compute_o3_dukm_full(ds):
    """
    O3 (vmr, mol/mol) -> DU/km using co-located T, plev.
    DU/km = 26.95560296256018 * [ P(hPa) / T(K) ] * O3_ppmv
    """
    if ('O3' not in ds) or ('T' not in ds) or ('plev' not in ds):
        raise KeyError("Dataset missing O3, T, or plev for DU/km calc.")
    O3 = ds['O3']
    T  = ds['T']
    p_hPa = plev_to_hPa(ds['plev'])  # (plev)
    O3_ppmv = O3 * 1e6
    COEFF = 26.95560296256018
    o3_dukm = COEFF * p_hPa * O3_ppmv / T
    o3_dukm.attrs['units'] = 'DU/km'
    o3_dukm.attrs['long_name'] = 'O3 local column density per km'
    return o3_dukm


def calc_parc_o3(var, p_top=30, p_bottom=70):
    """
    参考实现（与你之前一致）: 压力积分到 DU
    """
    m_air = 28.964 / (6.022E23)  # g / molecule
    g = 980.6                    # cm/s^2

    if 'plev' in var.dims:
        plev = var.plev; level = 'plev'
    elif 'lev' in var.dims:
        plev = var.lev;  level = 'lev'
    elif 'level' in var.dims:
        plev = var.level; level = 'level'
    else:
        raise ValueError('No pressure level coordinate found in data.')

    time = var.time
    delta_p = np.zeros((len(time), len(plev)))

    if plev[0] < plev[-1] and plev[-1] <= 1000:
        factor = 100; factor_2 = 1
    elif plev[0] > plev[-1] and plev[0] <= 1000:
        factor = 100; factor_2 = 1
    elif plev[0] < plev[-1] and plev[-1] > 1000:
        factor = 1;   factor_2 = 100
    elif plev[0] > plev[-1] and plev[0] > 1000:
        factor = 1;   factor_2 = 100
    else:
        factor = 1;   factor_2 = 100

    if plev[0] < plev[-1]:
        for i in range(1, len(plev)):
            delta_p[:, i] = plev[i] - plev[i-1]
        weights_p = xr.DataArray(delta_p * factor, dims=['time', level], coords=[time, plev])
        O3 = var * weights_p * 10 / (g * m_air)
        O3 = O3.sel(**{level: slice(p_top * factor_2, p_bottom * factor_2)})
        O3 = O3.sum(dim=level) / 2.687E16
    else:
        for i in range(0, len(plev)-1):
            delta_p[:, i] = plev[i] - plev[i+1]
        weights_p = xr.DataArray(delta_p * factor, dims=['time', level], coords=[time, plev])
        O3 = var * weights_p * 10 / (g * m_air)
        O3 = O3.sel(**{level: slice(p_bottom * factor_2, p_top * factor_2)})
        O3 = O3.sum(dim=level) / 2.687E16

    return O3.where(O3 != 0)


def _log_pressure_edges_ln(p_Pa):
    lp = np.log(p_Pa.astype(float))
    L = lp.size
    lp_edge = np.empty(L + 1, dtype=float)
    lp_edge[1:-1] = 0.5 * (lp[:-1] + lp[1:])
    lp_edge[0]  = lp[0]  + 0.5 * (lp[0]  - lp[1])
    lp_edge[-1] = lp[-1] + 0.5 * (lp[-1] - lp[-2])
    return lp_edge


def du_from_dukm(ds, p_top=30, p_bottom=70):
    """
    用 DU/km 反积分得到给定层的 DU：DU = Σ (DU/km × Δz)
    Δz_km = (R_d/g) * T * Δln p / 1000
    """
    dukm = compute_o3_dukm_full(ds)  # (time,plev,lat,lon)
    T    = ds['T']

    p_hPa = plev_to_hPa(ds['plev']).values
    p_Pa  = p_hPa * 100.0
    lp_edge = _log_pressure_edges_ln(p_Pa)
    dln = np.abs(lp_edge[:-1] - lp_edge[1:])
    dln_da = xr.DataArray(dln, dims=['plev'], coords={'plev': ds['plev']})

    pmin, pmax = min(p_top, p_bottom), max(p_top, p_bottom)
    mask = xr.DataArray(((p_hPa >= pmin) & (p_hPa <= pmax)),
                        dims=['plev'], coords={'plev': ds['plev']})

    R_d = 287.05
    g0  = 9.80665
    dz_km = (R_d / g0) * T * dln_da / 1000.0
    dz_km = dz_km.where(mask)

    du = (dukm * dz_km).sum(dim='plev')  # (time,lat,lon) [DU]
    du.attrs['units'] = 'DU'
    du.attrs['long_name'] = f'Partial O3 column ({pmin}-{pmax} hPa) from DU/km'
    return du


# ============= 60–90°N 余弦纬权平均 & 最小值（时间、数值） =============

def coslat_mean_60_90(da):
    """
    对输入 DataArray(… , lat[, lon]) 做：
      1) 先对经度平均（若存在 lon 维）
      2) 再在 60–90°N 做 cos(lat) 加权平均
    返回去除了 lat/lon 的 DataArray
    """
    if 'lon' in da.dims:
        da = da.mean(dim='lon')
    lat = da['lat']
    # 方向鲁棒选择
    if lat[0] <= lat[-1]:
        sub = da.sel(lat=slice(60, 90))
    else:
        sub = da.sel(lat=slice(90, 60))
    w = np.cos(np.deg2rad(sub['lat']))
    return sub.weighted(w).mean(dim='lat')


def verify_60_90_minimum(ds, tag, p_top=30, p_bottom=70):
    """
    计算 60–90°N 带的 DU 时间序列（两种方法）并打印最小值与一致性统计。
    """
    print(f"\n[VERIFY-60_90] {tag}: 60–90°N partial DU ({p_top}-{p_bottom} hPa) — pressure vs. DU/km reintegration")

    # 常数核对
    k_B = 1.380649e-23
    DU_m2 = 2.687e20
    C_Pa = 1000.0 / (k_B * DU_m2)
    C_hPa_ppmv = C_Pa * 100.0 * 1e-6
    print(f"  COEFF_theory(hPa-ppmv) = {C_hPa_ppmv:.12f};  COEFF_code = 26.955602962560")

    # A) 压力积分得到 DU 场，再做 60–90°N 加权平均
    du_press = calc_parc_o3(ds['O3'], p_top, p_bottom)    # (time,lat,lon)
    ts_press = coslat_mean_60_90(du_press)                # (time)

    # B) 由 DU/km 反积分到 DU 场，再做 60–90°N 加权平均
    du_reint = du_from_dukm(ds, p_top, p_bottom)          # (time,lat,lon)
    ts_reint = coslat_mean_60_90(du_reint)                # (time)

    # 时间序列一致性
    a = ts_press.values
    b = ts_reint.values
    mask = np.isfinite(a) & np.isfinite(b)
    if not np.any(mask):
        print("  [WARN] No finite values for comparison in 60–90°N.")
        return

    a, b = a[mask], b[mask]
    diff = b - a
    mae  = float(np.mean(np.abs(diff)))
    rmse = float(np.sqrt(np.mean(diff**2)))
    mean_a = float(np.mean(a))
    mean_b = float(np.mean(b))
    mape = float(100.0 * np.mean(np.abs(diff) / np.maximum(1e-9, np.abs(a))))
    corr = float(np.corrcoef(a, b)[0, 1]) if a.size >= 2 else np.nan

    # 最小值（指标）与发生时间
    tcoord = ts_press['time'].values[mask]
    iamin = int(np.argmin(a)); ibmin = int(np.argmin(b))
    amin, atime = float(a[iamin]), tcoord[iamin]
    bmin, btime = float(b[ibmin]), tcoord[ibmin]

    print(f"  Mean(DU) pressure  : {mean_a:.3f} DU")
    print(f"  Mean(DU) from DU/km: {mean_b:.3f} DU")
    print(f"  MAE={mae:.3f} DU, RMSE={rmse:.3f} DU, MAPE={mape:.2f} %, r={corr:.5f}")
    print(f"  MIN pressure  : {amin:.3f} DU at {np.datetime_as_string(atime, unit='D')}")
    print(f"  MIN fromDU/km : {bmin:.3f} DU at {np.datetime_as_string(btime, unit='D')}")
    print(f"  ΔMIN (fromDU/km - pressure) = {bmin - amin:+.3f} DU")

    # 返回两个时间序列（如你后续要画图/保存）
    return ts_press, ts_reint


# =========================================================
# Plotting / GIF routines（保持不变）
# =========================================================

def plot_spliced_seasonal_average(datasets, lat, height_km, output_file):
    lat_e = compute_edges(lat)
    h_e   = compute_edges(height_km.values)
    LAT_e, HT_e = np.meshgrid(lat_e, h_e)
    LATc, HTc   = np.meshgrid(lat, height_km.values)

    tags = list(datasets.keys())

    fig, axes = plt.subplots(2, 3, figsize=(15, 10),
                             sharex=True, sharey=True)
    axes = axes.flatten()

    pcm = None
    for i, tag in enumerate(tags):
        ds = datasets[tag]
        o3_dukm = compute_o3_dukm_full(ds)             # (time,plev,lat,lon)
        o3_mean = o3_dukm.mean(dim=['time', 'lon'])     # (plev,lat)
        u_mean  = ds['U'].mean(dim=['time', 'lon'])     # (plev,lat)

        ax = axes[i]
        pcm = ax.pcolormesh(LAT_e, HT_e, o3_mean.values,
                            shading='auto', cmap='rainbow')
        ct  = ax.contour(LATc, HTc, u_mean.values,
                         levels=[20, 30, 40, 50],
                         colors='white', linewidths=1)
        ax.clabel(ct, fmt='%d', inline=True, fontsize=8)
        ax.set_title(f'({chr(97+i)}) {tag}')
        ax.set_xlabel('Latitude (°)')
        ax.set_ylabel('Height (km)')
        ax.set_ylim(0, 30)
        for spine in ax.spines.values():
            spine.set_linewidth(0.8)

    if len(tags) < len(axes):
        for k in range(len(tags), len(axes)):
            fig.delaxes(axes[k])

    cax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
    cb = fig.colorbar(pcm, cax=cax, orientation='vertical')
    cb.set_label('O₃ [DU/km]')

    plt.tight_layout(rect=[0, 0, 0.9, 1])
    fig.savefig(output_file, dpi=300)
    plt.close(fig)


def create_o3_gif(ds, lat, height_km, frames_dir, gif_file, duration=0.5):
    os.makedirs(frames_dir, exist_ok=True)
    lat_e = compute_edges(lat)
    h_e   = compute_edges(height_km.values)
    LAT_e, HT_e = np.meshgrid(lat_e, h_e)
    LATc, HTc   = np.meshgrid(lat, height_km.values)

    o3_dukm_full = compute_o3_dukm_full(ds)
    o3_dukm_lonmean = o3_dukm_full.mean(dim='lon')

    vmin = float(np.nanmin(o3_dukm_lonmean.values))
    vmax = float(np.nanmax(o3_dukm_lonmean.values))

    writer = imageio.get_writer(gif_file, mode='I', duration=duration)

    for i, t in enumerate(ds.time.values):
        o3_now = o3_dukm_lonmean.isel(time=i)
        u_now  = ds['U'].isel(time=i).mean(dim='lon')

        fig, ax = plt.subplots(figsize=(8, 6))
        pcm = ax.pcolormesh(LAT_e, HT_e, o3_now.values,
                            shading='auto', cmap='rainbow',
                            vmin=vmin, vmax=vmax)
        ct = ax.contour(LATc, HTc, u_now.values,
                        levels=[20, 30, 40, 50],
                        colors='white', linewidths=1)
        ax.clabel(ct, fmt='%d', inline=True, fontsize=8)
        ax.set_title(str(t))
        ax.set_xlabel('Latitude (°)')
        ax.set_ylabel('Height (km)')
        ax.set_ylim(0, 30)
        for spine in ax.spines.values():
            spine.set_linewidth(0.8)
        cax = fig.add_axes([0.90, 0.15, 0.02, 0.7])
        cb = fig.colorbar(pcm, cax=cax, orientation='vertical')
        cb.set_label('O₃ [DU/km]')

        frame = os.path.join(frames_dir, f'frame_{i:03d}.png')
        fig.savefig(frame, dpi=200)
        plt.close(fig)
        writer.append_data(imageio.imread(frame))

    writer.close()


def create_combined_gif(datasets, lat, height_km, frames_dir, combo_file, duration=0.5):
    os.makedirs(frames_dir, exist_ok=True)
    lat_e = compute_edges(lat)
    h_e   = compute_edges(height_km.values)
    LAT_e, HT_e = np.meshgrid(lat_e, h_e)
    LATc, HTc   = np.meshgrid(lat, height_km.values)

    tags = list(datasets.keys())
    dukm_lonmean = {}
    for tag, ds in datasets.items():
        o3_dukm_full = compute_o3_dukm_full(ds)
        dukm_lonmean[tag] = o3_dukm_full.mean(dim='lon')

    times = next(iter(datasets.values())).time.values
    all_vals = np.concatenate([dukm_lonmean[tag].values.reshape(-1) for tag in tags])
    vmin = float(np.nanmin(all_vals))
    vmax = float(np.nanmax(all_vals))
    writer = imageio.get_writer(combo_file, mode='I', duration=duration)

    for ti, t in enumerate(times):
        fig, axes = plt.subplots(2, 3, figsize=(15, 10), sharex=True, sharey=True)
        axes = axes.flatten()
        pcm = None
        for j, tag in enumerate(tags):
            ds   = datasets[tag]
            o3_2d = dukm_lonmean[tag].isel(time=ti)
            u2d   = ds['U'].isel(time=ti).mean(dim='lon')

            ax = axes[j]
            pcm = ax.pcolormesh(LAT_e, HT_e, o3_2d.values,
                                shading='auto', cmap='rainbow',
                                vmin=vmin, vmax=vmax)
            ct  = ax.contour(LATc, HTc, u2d.values,
                             levels=[20, 30, 40, 50],
                             colors='white', linewidths=1)
            ax.clabel(ct, fmt='%d', inline=True, fontsize=8)
            ax.set_title(f'({chr(97+j)}) {tag}')
            ax.set_xlabel('Latitude (°)')
            ax.set_ylabel('Height (km)')
            ax.set_ylim(0, 30)
            for spine in ax.spines.values():
                spine.set_linewidth(0.8)

        if len(tags) < len(axes):
            for k in range(len(tags), len(axes)):
                fig.delaxes(axes[k])

        cax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
        cb = fig.colorbar(pcm, cax=cax, orientation='vertical')
        cb.set_label('O₃ [DU/km]')
        fig.suptitle(str(t))
        plt.tight_layout(rect=[0, 0, 0.9, 0.95])

        frame = os.path.join(frames_dir, f'combo_{ti:03d}.png')
        fig.savefig(frame, dpi=250)
        plt.close(fig)
        writer.append_data(imageio.imread(frame))

    writer.close()


def create_spliced_difference_gif(datasets, lat, height_km, frames_dir, gif_file, duration=0.5):
    os.makedirs(frames_dir, exist_ok=True)
    lat_e = compute_edges(lat)
    h_e   = compute_edges(height_km.values)
    LAT_e, HT_e = np.meshgrid(lat_e, h_e)
    LATc, HTc   = np.meshgrid(lat, height_km.values)

    exp_tags = [t for t in datasets if t != 'Reference']
    dukm_lonmean = {}
    for tag, ds in datasets.items():
        o3_dukm_full = compute_o3_dukm_full(ds)
        dukm_lonmean[tag] = o3_dukm_full.mean(dim='lon')

    ref_dukm = dukm_lonmean['Reference']
    times = datasets[exp_tags[0]].time.values

    diffs_flat = np.concatenate([
        (dukm_lonmean[tag] - ref_dukm).values.reshape(-1) for tag in exp_tags
    ])
    max_abs = float(np.nanmax(np.abs(diffs_flat)))
    if not np.isfinite(max_abs) or max_abs == 0: max_abs = 1.0
    vmin, vmax = -max_abs, max_abs

    writer = imageio.get_writer(gif_file, mode='I', duration=duration)

    for ti, t in enumerate(times):
        fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True, sharey=True)
        axes = axes.flatten()
        pcm = None
        for j, tag in enumerate(exp_tags):
            ds     = datasets[tag]
            diff2d = (dukm_lonmean[tag].isel(time=ti) - ref_dukm.isel(time=ti))
            u2d    = ds['U'].isel(time=ti).mean(dim='lon')

            ax = axes[j]
            pcm = ax.pcolormesh(LAT_e, HT_e, diff2d.values,
                                shading='auto', cmap='RdBu_r',
                                vmin=vmin, vmax=vmax)
            ct  = ax.contour(LATc, HTc, u2d.values,
                             levels=[20, 30, 40, 50],
                             colors='white', linewidths=1)
            ax.clabel(ct, fmt='%d', inline=True, fontsize=8)
            ax.set_title(f'({chr(97+j)}) {tag} – Reference')
            ax.set_xlabel('Latitude (°)')
            ax.set_ylabel('Height (km)')
            ax.set_ylim(0, 30)
            for spine in ax.spines.values():
                spine.set_linewidth(0.8)

        cax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
        cb = fig.colorbar(pcm, cax=cax, orientation='vertical')
        cb.set_label('ΔO₃ [DU/km]')
        fig.suptitle(str(t))
        plt.tight_layout(rect=[0, 0, 0.9, 0.95])

        frame = os.path.join(frames_dir, f'diff_{ti:03d}.png')
        fig.savefig(frame, dpi=250)
        plt.close(fig)
        writer.append_data(imageio.imread(frame))

    writer.close()


def create_snapshot_panel(datasets, lat, height_km, tag, out_png):
    ds0 = datasets[tag].isel(time=0)
    o3_dukm_full = compute_o3_dukm_full(datasets[tag])
    o3_now = o3_dukm_full.isel(time=0).mean(dim='lon')
    U_lon  = ds0['U'].mean(dim='lon')

    lat_e = compute_edges(lat)
    h_e   = compute_edges(height_km.values)
    LAT_e, HT_e = np.meshgrid(lat_e, h_e)
    LATc, HTc   = np.meshgrid(lat, height_km.values)

    fig, ax = plt.subplots(figsize=(10, 6))
    pcm = ax.pcolormesh(LAT_e, HT_e, o3_now.values,
                        shading='auto', cmap='rainbow')
    ct = ax.contour(LATc, HTc, U_lon.values,
                    levels=[20, 30, 40, 50],
                    colors='white', linewidths=1)
    ax.clabel(ct, fmt='%d', inline=True, fontsize=8)
    ax.set_xlabel('Latitude (°)')
    ax.set_ylabel('Height (km)')
    ax.set_ylim(0, 30)
    for spine in ax.spines.values():
        spine.set_linewidth(0.8)
    cbar = fig.colorbar(pcm, ax=ax, orientation='vertical', pad=0.02)
    cbar.set_label('O₃ [DU/km]')
    fig.tight_layout()
    fig.savefig(out_png, dpi=300)
    plt.close(fig)


# =========================================================
# Main workflow
# =========================================================

if __name__ == "__main__":
    # --- choose period ---
    year  = '0008'
    start, end = "0008-03-01", "0008-05-31"

    # --- output dir ---
    PARENT = "/home/weiji/restart_exam/plots/O3_DUkm"
    os.makedirs(PARENT, exist_ok=True)

    # --- styling ---
    set_plot_style()

    # --- data patterns ---
    patterns = [
        (f"{year}_Feb_couple",
         f"/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/"
         f"BWCN.e122.f19_g16.002_{year}/Feb/"
         f"BWCN.e122.f19_g16.002.{year}-02.*.nc"),
        (f"{year}_Mar_couple",
         f"/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/"
         f"BWCN.e122.f19_g16.002_{year}/Mar/"
         f"BWCN.e122.f19_g16.002.{year}-03.*.nc"),
        (f"{year}_Feb_nocouple",
         f"/home/weiji/restart_exam/nocouple_data/{year}/Feb_NOCOUPL/"
         f"{year}-02/*.nc"),
        (f"{year}_Mar_nocouple",
         f"/home/weiji/restart_exam/nocouple_data/{year}/Mar_NOCOUPL/"
         f"{year}-03/*.nc"),
    ]

    control_file = (
        "/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/"
        "BWCN.e122.f19_g16.002/"
        "BWCN.e122.f19_g16.002.cam.h3.0001-0023.int.nc"
    )

    # --- load ---
    datasets = { tag: load_hindcast(pat, start, end) for tag, pat in patterns }
    datasets['Reference'] = load_control(control_file, start, end)

    # --- coords ---
    example   = next(iter(datasets.values()))
    lat       = example.lat.values
    plev      = example.plev
    height_km = pressure_to_height(plev)

    # ========== NEW: 60–90°N DU 时间序列 & 最小值验证 ==========
    print("\n========== 60–90°N DU verification (30–70 hPa) ==========")
    ts_store = {}
    for tag, ds in datasets.items():
        out = verify_60_90_minimum(ds, tag, p_top=30, p_bottom=70)
        if out is not None:
            ts_press, ts_reint = out
            ts_store[tag] = {'pressure': ts_press, 'from_dukm': ts_reint}

    # ======= 以下原有绘图流程保持不变，可按需保留/注释 =======
    plot_spliced_seasonal_average(
        datasets, lat, height_km,
        os.path.join(PARENT, "MAM_average_spliced.png")
    )

    for tag, ds in datasets.items():
        scen_dir = os.path.join(PARENT, tag)
        frames   = os.path.join(scen_dir, "frames")
        gif_file = os.path.join(scen_dir, f"O3_{tag}.gif")
        os.makedirs(scen_dir, exist_ok=True)
        create_o3_gif(ds, lat, height_km, frames, gif_file)

    combo_frames = os.path.join(PARENT, "combined_frames")
    combo_gif    = os.path.join(PARENT, "O3_combined.gif")
    create_combined_gif(datasets, lat, height_km, combo_frames, combo_gif, duration=0.5)

    diff_frames = os.path.join(PARENT, "diff_frames")
    diff_gif    = os.path.join(PARENT, "O3_diff_combined.gif")
    create_spliced_difference_gif(datasets, lat, height_km, diff_frames, diff_gif, duration=0.5)

    tag0 = f"{year}_Feb_couple"
    snapshot_png = os.path.join(PARENT, f"{tag0}_snapshot_t0.png")
    create_snapshot_panel(datasets, lat, height_km, tag0, snapshot_png)
